In [ ]:
# Run muna to before magstart ulit sa bagong training

import tensorflow as tf
from tensorflow.keras import backend as K
import gc

# Clear Keras session
K.clear_session()

# Force garbage collection
gc.collect()

0

In [1]:
# Renaming files

import os
import random

# List of all target folder paths
folder_paths = [
    'C:\\Users\\Aldrea Timotei\\Downloads\\Tomato\\Tomato\\Bacterial Spot',
    'C:\\Users\\Aldrea Timotei\\Downloads\\Tomato\\Tomato\\Healthy',
    'C:\\Users\\Aldrea Timotei\\Downloads\\Tomato\\Tomato\\Leaf Mold',
    'C:\\Users\\Aldrea Timotei\\Downloads\\Tomato\\Tomato\\Septoria Leaf Spot',
    'C:\\Users\\Aldrea Timotei\\Downloads\\Tomato\\Tomato\\Yellow Leaf Curl Virus'
]

for folder_path in folder_paths:
    files = os.listdir(folder_path)
    files = [f for f in files if os.path.isfile(os.path.join(folder_path, f))]

    # Step 1: Rename all to temporary names to avoid conflicts
    for i, filename in enumerate(files):
        temp_name = f"temp_{i}{os.path.splitext(filename)[1]}"
        os.rename(
            os.path.join(folder_path, filename),
            os.path.join(folder_path, temp_name)
        )

    # Step 2: Shuffle the temp files and rename to randomized numeric names
    temp_files = os.listdir(folder_path)
    temp_files = [f for f in temp_files if f.startswith("temp_")]
    random.shuffle(temp_files)  # Shuffle file order

    for index, temp_file in enumerate(temp_files, start=1):
        extension = os.path.splitext(temp_file)[1]
        new_name = f"{index}{extension}"
        os.rename(
            os.path.join(folder_path, temp_file),
            os.path.join(folder_path, new_name)
        )

    print(f"Files in '{folder_path}' renamed randomly with numeric names starting from 1.")


Files in 'C:\Users\Aldrea Timotei\Downloads\Tomato\Tomato\Bacterial Spot' renamed randomly with numeric names starting from 1.
Files in 'C:\Users\Aldrea Timotei\Downloads\Tomato\Tomato\Healthy' renamed randomly with numeric names starting from 1.
Files in 'C:\Users\Aldrea Timotei\Downloads\Tomato\Tomato\Leaf Mold' renamed randomly with numeric names starting from 1.
Files in 'C:\Users\Aldrea Timotei\Downloads\Tomato\Tomato\Septoria Leaf Spot' renamed randomly with numeric names starting from 1.
Files in 'C:\Users\Aldrea Timotei\Downloads\Tomato\Tomato\Yellow Leaf Curl Virus' renamed randomly with numeric names starting from 1.


In [5]:
# Image Resizing for Multiple Folders (256x256)

import os
from PIL import Image

# List of folders to resize images from
folders = [
    'C:/Users/Aldrea Timotei/Downloads/Tomato2/Tomato/Bacterial Spot',
    'C:/Users/Aldrea Timotei/Downloads/Tomato2/Tomato/Healthy',
    'C:/Users/Aldrea Timotei/Downloads/Tomato2/Tomato/Leaf Mold',
    'C:/Users/Aldrea Timotei/Downloads/Tomato2/Tomato/Septoria Leaf Spot',
    'C:/Users/Aldrea Timotei/Downloads/Tomato2/Tomato/Yellow Leaf Curl Virus'
]

# Target image size
target_size = (256, 256)

for input_folder in folders:
    # Output folder will be the same as input (overwrite original images)
    output_folder = input_folder

    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            try:
                img_path = os.path.join(input_folder, filename)
                img = Image.open(img_path).convert("RGB")
                img_resized = img.resize(target_size)

                save_path = os.path.join(output_folder, filename)
                img_resized.save(save_path)

                print(f"Resized and saved: {save_path}")
            except Exception as e:
                print(f"Failed to process {filename} in {input_folder}: {e}")


In [ ]:
# Augmentation

import os
import cv2
import albumentations as A
from glob import glob

# Define your 5 folders here (update paths as needed)
folders = [
    'C:/Users/Aldrea Timotei/Downloads/Tomato/Test/Yellow Leaf Curl Virus'
]

# Define augmentations (each applied separately)
augmentations = {
    "rotated": A.Rotate(limit=40, p=1.0),
    "zoomed": A.RandomScale(scale_limit=0.3, p=1.0),
    "cropped": A.RandomCrop(width=150, height=150, p=1.0),
    "vert_flipped": A.VerticalFlip(p=1.0),
    "brightened": A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.0, p=1.0),
}

def augment_and_save(image_path, folder, aug_name, aug):
    # Read image with OpenCV (BGR)
    image = cv2.imread(image_path)
    if image is None:
        print(f"Skipping unreadable image: {image_path}")
        return
    
    # Apply augmentation
    augmented = aug(image=image)
    aug_image = augmented["image"]

    # Build save path with suffix
    base_name = os.path.basename(image_path)
    name, ext = os.path.splitext(base_name)
    new_name = f"{name}_{aug_name}{ext}"
    save_path = os.path.join(folder, new_name)

    # Save augmented image
    cv2.imwrite(save_path, aug_image)

for folder in folders:
    print(f"Processing folder: {folder}")
    image_files = glob(os.path.join(folder, "*.*"))  # all files, you can filter extensions if needed

    for img_path in image_files:
        for aug_name, aug in augmentations.items():
            augment_and_save(img_path, folder, aug_name, aug)

    print(f"Completed augmentations in {folder}\n")

print("All augmentations done!")


Processing folder: C:/Users/Aldrea Timotei/Downloads/Tomato/Test/Yellow Leaf Curl Virus
Completed augmentations in C:/Users/Aldrea Timotei/Downloads/Tomato/Test/Yellow Leaf Curl Virus

All augmentations done!


In [ ]:
# Paghihiwalay ng Test Set

import os
import random
import shutil
from pathlib import Path

# --- SETTINGS ---
source_root = "C:\\Users\\Aldrea Timotei\\Downloads\\Tomato\\Tomato DS"     # Your dataset folder (e.g., "dataset/train")
test_root = "C:\\Users\\Aldrea Timotei\\Downloads\\Tomato\\test"                       # Output test folder
images_per_class = 100

# --- MAIN LOGIC ---
os.makedirs(test_root, exist_ok=True)

# Loop through each class folder
for class_name in os.listdir(source_root):
    class_path = os.path.join(source_root, class_name)
    if not os.path.isdir(class_path):
        continue

    # Create corresponding class folder inside test/
    test_class_path = os.path.join(test_root, class_name)
    os.makedirs(test_class_path, exist_ok=True)

    # List all images, shuffle, pick 100
    all_images = os.listdir(class_path)
    selected_images = random.sample(all_images, images_per_class)

    for img in selected_images:
        src = os.path.join(class_path, img)
        dst = os.path.join(test_class_path, img)
        shutil.move(src, dst)  # ✅ Move (removes from source)


print("✅ Test set created successfully with 100 images per class.")


✅ Test set created successfully with 100 images per class.
